In [1]:
import sys
import numpy as np

# sys.path.append('../../../src/')
from Rain.Rain import Rain
# sys.path.pop()

from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-05 22:45:22.843000: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-05 22:45:23.527325: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "lazy",
      "params": {
        "num_of_workers": 3,
        "ips": ['127.0.0.1', '127.0.0.1', '172.190.224.121'],
        "ports": [50151, 50152, 50160]
      }
    },
  "temp_data_path": "../../../",
  "partitions": 3,
  "iterations": 3,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 20,
    "batch_size": 16,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/breast_cancer/train_data.npy"), np.load(
        "../../../data/breast_cancer/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/breast_cancer/test_data.npy"), np.load(
        "../../../data/breast_cancer/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 30
    num_labels = 2
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()
X_train = np.reshape(X_train, [-1, 30])
y_train = to_categorical(y_train)

In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-05 22:45:24,481 [ERROR] [Rain] Error in the config: Error in partitions: argument of type 'int' is not iterable
2023-07-05 22:45:24,482 [DEBUG] [Rain] Rain is initialized
2023-07-05 22:45:24,483 [DEBUG] [Provisioner] Creating coordinator
2023-07-05 22:45:24,484 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
2023-07-05 22:45:24,485 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-05 22:45:24,486 [DEBUG] [LazyProvisioner] Provisioner is initialized
2023-07-05 22:45:24,488 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-05 22:45:24,489 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-05 22:45:24,491 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


In [8]:
# model = rain.train(X_train, y_train, strategy='async')

In [9]:
# X_test, y_test = get_test_data()
# X_test = np.reshape(X_test, [-1, 30])
# y_test = to_categorical(y_test)
# loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
# print("\nTest accuracy: %.1f%%" % (100.0 * acc))

In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-05 22:45:24,512 [INFO] [Provisioner] provisioner is serving
2023-07-05 22:45:24,513 [DEBUG] [Provisioner] Starting coordinator
2023-07-05 22:45:24,514 [INFO] [Coordinator] coordinator is serving
2023-07-05 22:45:24,516 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-05 22:45:24,520 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-05 22:45:24,521 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-05 22:45:24,522 [DEBUG] [LazyProvisioner] Creating 3 workers
2023-07-05 22:45:24,523 [DEBUG] [Provisioner] [Created workers]
IPs : ['127.0.0.1', '127.0.0.1', '172.190.224.121'], ports: [50151, 50152, 50160], statuses: [1, 1, 1], IDs : [1, 2, 3]
2023-07-05 22:45:24,525 [DEBUG] [DividerAmbassador] divider ambassador is serving
2023-07-05 22:45:24,526 [DEBUG] [DividerProxy] Training Started
2023-07-05 22:45:24,529 [DEBUG] [Provisioner] Received '' from the co

In [11]:
X_test, y_test = get_test_data()
X_test = np.reshape(X_test, [-1, 30])
y_test = to_categorical(y_test)
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

8/8 [==============================] - 0s 1ms/step - loss: 0.2569 - accuracy: 0.9386

Test accuracy: 93.9%


: 